# Solution E3 · The same pruning prunes nothing: `ccp_alpha` depends on the units of the target

**Type**: Find the bug

## Objective

Find out why the same `ccp_alpha=5e-4` that gave 182 leaves on prices in hundreds of thousands of dollars leaves **2453 leaves** when the prices are in dollars: cost-complexity pruning uses the impurity (the squared error) in units of the target, so `alpha` has to be rescaled with the variance of `y`.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
y_tr_dollars, y_te_dollars = y_tr * 100000, y_te * 100000


def tree(alpha, y_train):
    return DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=alpha).fit(X_tr, y_train)

## Commented solution

In [ ]:
t_ref = tree(5e-4, y_tr)                                         # target in hundreds of thousands of dollars
t_bug = tree(5e-4, y_tr_dollars)                                  # same alpha, target in dollars
ref_leaves, bug_leaves = t_ref.get_n_leaves(), t_bug.get_n_leaves()
ref_rmse = mean_squared_error(y_te, t_ref.predict(X_te), squared=False)
bug_rmse = mean_squared_error(y_te_dollars, t_bug.predict(X_te), squared=False) / 100000
print(f"leaves: {ref_leaves} (target in hundreds of thousands) against {bug_leaves} (in dollars) | RMSE: {ref_rmse:.3f} against {bug_rmse:.3f}")

**Where the bug is.** Cost-complexity pruning removes a branch if the impurity reduction it brings is smaller than `alpha` per leaf. The impurity of a regression tree is the squared error **in units of the target squared**: with `y` in dollars it is 10¹⁰ times larger. The same `alpha = 5e-4` is therefore worth
10⁻¹⁴ of the variance of the target: no branch is cut and the tree stays the complete one.

In [ ]:
scale_factor = 100000 ** 2
alpha_dollars = 5e-4 * scale_factor
t_ok = tree(alpha_dollars, y_tr_dollars)
ok_leaves = t_ok.get_n_leaves()
ok_rmse = mean_squared_error(y_te_dollars, t_ok.predict(X_te), squared=False) / 100000
print(f"equivalent alpha for the target in dollars: {alpha_dollars:.1e} -> {ok_leaves} leaves, RMSE {ok_rmse:.3f}")

In [ ]:
def relative_alpha(alpha, y):
    return alpha / np.var(y)


rel_alpha = relative_alpha(5e-4, y_tr)
print(f"relative alpha (fraction of the variance of the target): {rel_alpha:.2e} = {relative_alpha(alpha_dollars, y_tr_dollars):.2e} in dollars")

## Check

In [ ]:
with verify():
    check(ref_leaves < 250 and bug_leaves > 2000 and bug_rmse > ref_rmse + 0.005, "the colleague: 182 leaves against over 2000 (no pruning), with a worse RMSE", "don't modify the colleague's code")
    check(scale_factor == 1e10 and close(alpha_dollars, 5e6), "the scale factor is 1e10 (the square of 100000); equivalent alpha 5e6", "impurity = squared error: it scales like the square of the target")
    check(ok_leaves == ref_leaves and abs(ok_rmse - ref_rmse) < 1e-6, "with the rescaled alpha the number of leaves and the error match those on the original scale", "tree(alpha_dollars, y_tr_dollars)")
    check(close(relative_alpha(5e-4, y_tr), rel_alpha) and close(relative_alpha(alpha_dollars, y_tr_dollars), rel_alpha), "relative alpha (alpha / variance of the target) equal in the two units", "alpha / np.var(y)")
done()

## What you learned

- **`ccp_alpha` (regression) and `min_impurity_decrease` are in units of the target squared.** Changing units (dollars, thousands, logarithm) changes the pruning drastically and silently: with the same number, 182 leaves or 2453. The code raises no error and the RMSE gets worse only a little (0.653 against 0.640), so nobody notices.
- **It also holds if you transform the target:** moving to `log(y)` (for example for a price model) makes any `alpha` found before useless. If you need a transferable parameter, express it as a **fraction of the variance** (`alpha / var(y)`) and convert it back when you change scale.
- For classification the impurity (Gini, entropy) is already dimensionless (between 0 and 1), so `ccp_alpha` is comparable between problems; it is a case where the intuition "it holds for everyone" is right, but not for regression.
- Rule: **never copy a hyperparameter** from another problem or another unit without asking yourself in which units it is. If a hyperparameter has no effect (or a huge one), check something measurable (here: the number of leaves) and not only the final metric.

## Common pitfall

Copying a `ccp_alpha` from a colleague's notebook, from a tutorial or from a previous project. The number has no meaning outside the units of the target on which it was chosen.

## Interview question

<details><summary>Why can a hyperparameter found on one dataset fail on another, even a similar one?</summary>

Because many depend on the scale of the data or on the sample size: the ccp_alpha of a regression tree is in units of the target squared, the regularization of a logistic regression depends on the scale of the features and on n, the learning rate on the number of trees. I always search the hyperparameters on the current problem,
and when I transfer a value I re-express it in dimensionless form (for example as a fraction of the variance of the target).

</details>